In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Transformations
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# Load datasets with correct paths
train_dataset = datasets.ImageFolder(
    root="/kaggle/input/q1-stage-3-2026/PlantVillage/train",
    transform=train_transform
)
test_dataset = datasets.ImageFolder(
    root="/kaggle/input/q1-stage-3-2026/PlantVillage/test",
    transform=test_transform
)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Display some sample images
classes = train_dataset.classes
fig, axes = plt.subplots(2, 5, figsize=(12,5))
for i, ax in enumerate(axes.flatten()):
    img, label = train_dataset[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')
plt.show()


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class PotatoCNN(nn.Module):
    def __init__(self):
        super(PotatoCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.conv5 = nn.Conv2d(128, 128, 3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128*1*1, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN().to(device)


In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy


In [ ]:
import matplotlib.pyplot as plt

# Setup
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 15

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

# Plotting Loss and Accuracy
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import torch.optim as optim
import matplotlib.pyplot as plt

# Residual CNN Model
class PotatoCNNResidual(nn.Module):
    def __init__(self):
        super(PotatoCNNResidual, self).__init__()
        self.conv1 = nn.Conv2d(3,16,3,padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32,64,3,padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64,128,3,padding=1)
        self.bn4 = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(128,128,3,padding=1)
        self.bn5 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(128*1*1,128)
        self.fc2 = nn.Linear(128,3)

        # 1x1 conv to match channels for residual addition
        self.res_conv = nn.Conv2d(32,128,1)

    def forward(self,x):
        x1 = self.pool(F.relu(self.bn1(self.conv1(x))))  # Conv1
        x2 = self.pool(F.relu(self.bn2(self.conv2(x1)))) # Conv2
        x3 = self.pool(F.relu(self.bn3(self.conv3(x2)))) # Conv3
        x4 = self.pool(F.relu(self.bn4(self.conv4(x3)))) # Conv4

        # Residual connection: x2 -> x4
        x2_resized = self.res_conv(F.adaptive_avg_pool2d(x2,(x4.size(2), x4.size(3))))
        x4 = x4 + x2_resized

        x5 = self.pool(F.relu(self.bn5(self.conv5(x4)))) # Conv5
        x_flat = x5.view(x5.size(0),-1)
        x_fc = F.relu(self.fc1(x_flat))
        out = self.fc2(x_fc)
        return out

# Training & Validation Functions
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return total_loss/len(dataloader), correct/total

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return total_loss/len(dataloader), correct/total

# Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
res_model = PotatoCNNResidual().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(res_model.parameters(), lr=0.001)
num_epochs = 15

train_losses_res, val_losses_res = [], []
train_accs_res, val_accs_res = [], []

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs} - Residual Model")

    train_loss, train_acc = train_one_epoch(res_model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(res_model, test_loader, criterion, device)

    train_losses_res.append(train_loss)
    val_losses_res.append(val_loss)
    train_accs_res.append(train_acc)
    val_accs_res.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

# Plot Training & Validation Metrics
plt.figure(figsize=(12,5))

# Loss curve
plt.subplot(1,2,1)
plt.plot(train_losses_res, label="Train Loss")
plt.plot(val_losses_res, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Residual Model Loss Curve")
plt.legend()

# Accuracy curve
plt.subplot(1,2,2)
plt.plot(train_accs_res, label="Train Acc")
plt.plot(val_accs_res, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Residual Model Accuracy Curve")
plt.legend()

plt.show()
